In [ ]:
!pip install ucimlrepo

In [ ]:
!pip install ucimlrepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from ucimlrepo import fetch_ucirepo

# ==========================================
# 1. Load Dataset
# ==========================================
try:
    # fetch dataset
    breast_cancer_wisconsin_original = fetch_ucirepo(id=15)

    # data (as pandas dataframes)
    X = breast_cancer_wisconsin_original.data.features
    y = breast_cancer_wisconsin_original.data.targets

    # print(breast_cancer_wisconsin_original.metadata)
    # print(breast_cancer_wisconsin_original.variables)

except Exception as e:
    print(f"UCI server failed to fetch (Error: {e}), using fallback CSV...")

    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/breast-cancer-wisconsin.data"

    columns = [
        "id","clump_thickness","uniformity_cell_size","uniformity_cell_shape",
        "marginal_adhesion","single_epithelial_cell_size","bare_nuclei",
        "bland_chromatin","normal_nucleoli","mitoses","class"
    ]

    df = pd.read_csv(url, names=columns)

    X = df.drop(["id", "class"], axis=1)
    y = df["class"]

UCI server failed to fetch (Error: Error connecting to server), using fallback CSV...


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from ucimlrepo import fetch_ucirepo


# Convert all columns to numeric (important fix)
X = X.apply(pd.to_numeric, errors='coerce')
y = y.apply(pd.to_numeric, errors='coerce')

# Combine and remove missing values
initial_rows_count = X.shape[0]
combined_df = pd.concat([X, y], axis=1)
cleaned_df = combined_df.dropna()

X = cleaned_df[X.columns]
y = cleaned_df[y.name].squeeze().astype(int)

# Convert labels properly
y = y.replace({2: -1, 4: 1})
print(f"Removed {initial_rows_count - X.shape[0]} rows containing NaN values.")

# ==========================================
# 2. Train-Test Split
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==========================================
# 3. Min-Max Normalization
# ==========================================
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================================
# 4. Train SVM (Sigmoid Kernel)
# ==========================================
clf = svm.SVC(
    kernel='sigmoid',
    gamma=1/X_train.shape[1],
    coef0=0,
    C=1.0
)

clf.fit(X_train, y_train)

print("Training completed")
print("Number of support vectors:", clf.support_vectors_.shape[0])

# ==========================================
# 5. Single Test Sample
# ==========================================
sample_index = 0

x_single = X_test[sample_index]
y_single = y_test.iloc[sample_index]

decision_score = clf.decision_function([x_single])

print("Ground truth:", y_single)
print("Decision score:", decision_score[0])

# ==========================================
# 6. Save Model Parameters
# ==========================================
np.savetxt("support_vectors_sigmoid.txt", clf.support_vectors_)
np.savetxt("dual_coef_sigmoid.txt", clf.dual_coef_)
np.savetxt("intercept_sigmoid.txt", clf.intercept_)
np.savetxt("xtest_sigmoid.txt", x_single)
np.savetxt("ytest_sigmoid.txt", [y_single])
np.savetxt("yclassificationscore_sigmoid.txt", decision_score)

print("All files saved successfully.")

# ==========================================
# 7. Model Evaluation
# ==========================================
y_pred = clf.predict(X_test)

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nAccuracy:", accuracy_score(y_test, y_pred))

Removed 16 rows containing NaN values.
Training completed
Number of support vectors: 81
Ground truth: 1
Decision score: 1.0933024324812444
All files saved successfully.

Confusion Matrix:
 [[78  1]
 [ 6 52]]

Classification Report:
               precision    recall  f1-score   support

          -1       0.93      0.99      0.96        79
           1       0.98      0.90      0.94        58

    accuracy                           0.95       137
   macro avg       0.95      0.94      0.95       137
weighted avg       0.95      0.95      0.95       137


Accuracy: 0.948905109489051


In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))

print(X.shape)
print(y.unique())

Precision: 0.9811320754716981
Recall: 0.896551724137931
F1-Score: 0.9369369369369369
(683, 9)
[-1  1]
